# Listing photos and views: Scheffé's comparisons

**Kruskal-Wallis Test with Multiple Comparisons** · a worked example from the StatNex teaching guide

**The story.** A resale app looked at 21 listings for the same kind of jacket, posted with 1, 3, 5 or 10 photos, and counted how many people viewed each listing in its first week.

**The question.** Do listings with more photos get more views, and how many photos are enough?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/kruskal_wallis.csv` (6 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
1 photo,3 photos,5 photos,10 photos
88.0,118.0,135.0,148.0
102.0,131.0,150.0,170.0
95.0,109.0,128.0,139.0
120.0,140.0,162.0,165.0
76.0,,141.0,158.0
110.0,,155.0,

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per level. Text and blank cells are skipped, so levels may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["1 photo", "3 photos", "5 photos", "10 photos"]
groups = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in levels]
sizes = np.array([len(g) for g in groups])

## Summary by level

In [ ]:
# All values are ranked together (ties share the average rank); each level's mean rank
# is what the test compares.
ranks = stats.rankdata(np.concatenate(groups))
mean_ranks = [part.mean() for part in np.split(ranks, np.cumsum(sizes)[:-1])]
summary = pd.DataFrame({"n": sizes, "mean rank": mean_ranks}, index=levels).T
print(summary)

## Kruskal-Wallis test

In [ ]:
# SciPy's H carries the tie correction; it is compared with chi-square on k - 1 df.
h, p = stats.kruskal(*groups)
df = len(groups) - 1
print(f"\nchi-square = {h:.6f}, df = {df}, P = {p:.6g}")

## Multiple comparison: Scheffé on mean ranks

In [ ]:
# Every pair of levels, on the mean ranks of the whole sample: the squared difference
# over its variance N(N + 1)/12 x (1/n1 + 1/n2) x C, where C = 1 - sum(t^3 - t)/(N^3 - N)
# corrects for ties, judged against chi-square on k - 1 df.
total = sizes.sum()
_, tie_sizes = np.unique(np.concatenate(groups), return_counts=True)
correction = 1 - (tie_sizes**3 - tie_sizes).sum() / (total**3 - total)
rows = []
for i, j in combinations(range(len(groups)), 2):
    variance = total * (total + 1) / 12 * (1 / sizes[i] + 1 / sizes[j]) * correction
    chi2 = (mean_ranks[i] - mean_ranks[j]) ** 2 / variance
    rows.append([levels[i], levels[j], chi2, stats.chi2.sf(chi2, df)])
pairs = pd.DataFrame(rows, columns=["level 1", "level 2", "chi-square", "P"])
print()
print(pairs.to_string(index=False))

## What the result says

The Kruskal-Wallis test compares groups by ranks, the positions of the listings when sorted by views. Its chi-square statistic is 14.89 with 3 degrees of freedom and the P value is 0.002. The P value is the chance of gaps this large if photos made no difference, so the photo counts differ somewhere.

Scheffé's method then compares every pair of photo counts. It is a cautious method: it protects against false alarms across all six pairs at once. Only two pairs pass. Ten photos beat one (P = 0.006) and five photos beat one (P = 0.040). For 1 against 3 photos P is 0.704, and for 5 against 10 photos P is 0.901.

So five or ten photos clearly get more views than one. A difference between three and one, or between ten and five, is not shown. That is not proof that they are equal: the listings are too few to settle it.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.